In [ ]:
!pip install requests pandas

In [ ]:
# ============================================================
# Step 2a: Scrape Coachella comments from PullPush
# ============================================================
import re, time, requests, pandas as pd
from datetime import datetime, timezone

BASE_URL = "https://api.pullpush.io/reddit/search/comment/"
HEADERS  = {"User-Agent": "Mozilla/5.0 Chrome/124.0.0.0 Safari/537.36",
             "Accept": "application/json"}

session = requests.Session()
session.headers.update(HEADERS)

# Fixed window aligned with search_data.csv
START_TS = int(datetime(2025, 4, 1,  tzinfo=timezone.utc).timestamp())
END_TS   = int(datetime(2025, 5, 31, tzinfo=timezone.utc).timestamp())
TARGET   = 100

def fetch_batch(q, before, seen_ids, subreddit=None):
    params = {"q": q, "after": START_TS, "before": before,
              "size": 100, "sort": "desc"}
    if subreddit:
        params["subreddit"] = subreddit
    try:
        resp = session.get(BASE_URL, params=params, timeout=20)
        resp.raise_for_status()
        items = resp.json().get("data", [])
    except Exception as e:
        print(f"  ⚠️  {e}"); return [], before

    rows = []; oldest = before
    for item in items:
        cid  = item.get("id", "")
        body = item.get("body", "").strip()
        if not body or body in ("[deleted]", "[removed]"): continue
        if cid in seen_ids: continue
        created = int(item.get("created_utc", 0))
        oldest  = min(oldest, created)
        seen_ids.add(cid)
        rows.append({
            "Date":         datetime.fromtimestamp(created, tz=timezone.utc).strftime("%Y-%m-%d"),
            "Keyword":      "Coachella",
            "Comment_Text": body.replace("\n", " "),
        })
    return rows, oldest

print("→ Fetching Coachella comments …")
coachella_rows = []; seen = set(); before = END_TS

for page in range(1, 10):
    if len(coachella_rows) >= TARGET: break
    batch, oldest = fetch_batch("Coachella", before, seen)
    coachella_rows.extend(batch)
    print(f"  Page {page}: +{len(batch)} | Total: {len(coachella_rows)}")
    if len(batch) < 100 or oldest >= before: break
    before = oldest
    time.sleep(3)

coachella_rows = coachella_rows[:TARGET]
print(f"\n✅ Coachella: {len(coachella_rows)} comments collected")
print(f"   Date range: {coachella_rows[0]['Date']} → {coachella_rows[-1]['Date']}")


→ Fetching Coachella comments …
  Page 1: +100 | Total: 100

✅ Coachella: 100 comments collected
   Date range: 2025-05-19 → 2025-05-16


In [ ]:
# ============================================================
# Step 2b: Scrape Alo Yoga comments (Fixed Dates & Keyword)
# ============================================================
import re, time, requests, pandas as pd
from datetime import datetime, timezone

# 🚨 BUG FIX
START_TS_ALO = int(datetime(2025, 4, 1, tzinfo=timezone.utc).timestamp())
END_TS_ALO   = int(datetime(2025, 5, 31, tzinfo=timezone.utc).timestamp())
TARGET_ALO   = 100

# ── Relevance filter ─────────────────────────────────────────
MUST_HAVE = [
    r'\balo yoga\b',
    r'\balo\b.{0,60}\baesthetic\b',
    r'\baesthetic\b.{0,60}\balo\b',
    r'\balo\b.{0,50}\b(legging|outfit|look|style|vibe|brand|overpriced|expensive|worth|gym|yoga|pilates|influencer|girlies|that girl|clean girl|athleisure|lululemon)\b',
    r'\b(outfit|look|style|vibe|brand|gym|yoga|pilates|influencer|that girl|clean girl|athleisure|lululemon)\b.{0,50}\balo\b',
]
EXCLUDE = [r'\bbirkin\b', r'\bchanel bag\b', r'\bdesigner bag\b',
           r'\bcarbon plate\b', r'\brunning shoe\b']

def is_relevant(text):
    t = text.lower()
    if not any(re.search(p, t) for p in MUST_HAVE): return False
    if any(re.search(p, t) for p in EXCLUDE):       return False
    if len(t.strip()) < 30:                          return False
    return True

# ── Queries ────────────────────────────
QUERIES = [
    {"q": "Alo Yoga"},
    {"q": "Alo aesthetic"},
    {"q": "Alo Yoga overpriced"},
    {"q": "Alo Yoga worth it"},
    {"q": "Alo", "subreddit": "lululemon"},
    {"q": "Alo", "subreddit": "gymsnark"},
    {"q": "Alo", "subreddit": "AloYoga"},
]

def fetch_batch_alo(q_cfg, before, seen_ids):
    params = {"after": START_TS_ALO, "before": before, "size": 100, "sort": "desc",
              **q_cfg}
    try:
        resp = session.get(BASE_URL, params=params, timeout=20)
        resp.raise_for_status()
        items = resp.json().get("data", [])
    except Exception as e:
        print(f"  ⚠️  {e}"); return [], before

    rows = []; oldest = before
    for item in items:
        cid  = item.get("id", "")
        body = item.get("body", "").strip()
        if not body or body in ("[deleted]", "[removed]"): continue
        if cid in seen_ids: continue
        if not is_relevant(body): continue
        created = int(item.get("created_utc", 0))
        oldest  = min(oldest, created)
        seen_ids.add(cid)
        rows.append({
            "Date":         datetime.fromtimestamp(created, tz=timezone.utc).strftime("%Y-%m-%d"),
            "Keyword":      "Alo Yoga",  # 🚨 BUG FIX
            "Comment_Text": body.replace("\n", " "),
        })
    return rows, oldest

print("→ Fetching Alo Yoga comments (with relevance filter) …")
alo_rows = []; seen_alo = set()

for q_cfg in QUERIES:
    if len(alo_rows) >= TARGET_ALO: break
    label  = q_cfg.get("subreddit", q_cfg.get("q", ""))
    before = END_TS_ALO
    print(f"  Query: {label}")

    for page in range(1, 8):
        if len(alo_rows) >= TARGET_ALO: break
        batch, oldest = fetch_batch_alo(q_cfg, before, seen_alo)
        alo_rows.extend(batch)
        print(f"    Page {page}: +{len(batch)} relevant | Total: {len(alo_rows)}")
        if len(batch) == 0 or oldest >= before: break
        before = oldest
        time.sleep(1.5)
    time.sleep(2)

alo_rows = alo_rows[:TARGET_ALO]
print(f"\n✅ Alo Yoga: {len(alo_rows)} relevant comments collected")

→ Fetching Alo Yoga comments (with relevance filter) …
  Query: Alo Yoga
    Page 1: +80 relevant | Total: 80
    Page 2: +63 relevant | Total: 143

✅ Alo Yoga: 100 relevant comments collected


In [ ]:
# ============================================================
# Step 2c: Merge Coachella + Alo Yoga → reddit_comments.csv
# ============================================================
import pandas as pd

df = pd.DataFrame(coachella_rows + alo_rows)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Keyword", "Date"]).reset_index(drop=True)

print("── Summary ──────────────────────────────────────────────────")
print(df.groupby("Keyword").size().rename("Comments").to_string())
print(f"\nDate range : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Total rows : {len(df)}")

print("\n── Sample (3 Coachella + 3 Alo) ────────────────────────────")
sample = pd.concat([
    df[df["Keyword"]=="Coachella"].head(3),
    df[df["Keyword"]=="Alo Yoga aesthetic"].head(3)
])
for _, row in sample.iterrows():
    print(f"[{row['Keyword'][:12]}] {row['Date'].date()} | {row['Comment_Text'][:100]}…")

df[["Date", "Keyword", "Comment_Text"]].to_csv(
    "reddit_comments.csv", index=False, encoding="utf-8-sig"
)
print("\n✅ Saved to 'reddit_comments.csv'")

# from google.colab import files; files.download("reddit_comments.csv")


── Summary ──────────────────────────────────────────────────
Keyword
Alo Yoga     100
Coachella    100

Date range : 2025-04-18 → 2025-05-19
Total rows : 200

── Sample (3 Coachella + 3 Alo) ────────────────────────────
[Coachella] 2025-05-16 | I have a friend who brings Salmiakki to the Finnish players on the Coachella Valley Firebirds team a…
[Coachella] 2025-05-16 | Ok I’m not trying to be mean but I saw pics of her today and thought the same thing. She was looking…
[Coachella] 2025-05-16 | # Q: Who's in the DJ booth? Does anybody else ever spin in there?  A: James Murphy (of DFA Records /…

✅ Saved to 'reddit_comments.csv'
